# EPRdating quick start

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/MiguelGamezL/EPRdating/blob/main/examples/quickstart.ipynb)

From EPR spectra to an age with its Monte Carlo uncertainty. The data are
simulated, so the notebook runs anywhere (Jupyter, JupyterLab, Google Colab).
Replace the simulation cells by your files (`read_epr`, `analyse_files`).

## Install

In **Google Colab** (or any fresh environment) run the next cell once. In a local
Jupyter with EPRdating already installed, skip it.

In [ ]:
%pip install "eprdating[plot]"

In [ ]:
import dataclasses

import matplotlib.pyplot as plt
import numpy as np

import eprdating
from eprdating import (
    History,
    LinearUptake,
    Sediment,
    ToothLayers,
    ToothSample,
    cosmic_dose_rate,
    fit_dose_response,
    plot,
)
from eprdating.spectra import (
    IntensityWindow,
    Spectrum,
    combined_intensity,
    field_for_g,
    intensity,
    pseudo_modulation,
)

print("EPRdating", eprdating.__version__)

## 1. A dose series of EPR spectra

With real data: `spectra = [read_epr(f) for f in files]` (Bruker `.DSC`/`.DTA`,
`.par`/`.spc`, ASCII or CSV). Here eight aliquots are simulated on a 50 mT
sweep at 9.43 GHz, with a true De of 40 Gy. The natural aliquot was
measured twice.

In [ ]:
rng = np.random.default_rng(1)
FREQ = 9.43                                           # GHz
B = np.linspace(311, 361, 512)                        # mT, a 50 mT sweep
c = field_for_g(2.0023, FREQ)                         # 336.49 mT

def line(c0, w):                                      # Lorentzian derivative
    x = (B - c0) / (w * np.sqrt(3) / 2)
    return -x / (1 + x**2) ** 2

shape = line(c + 0.40, 0.25) + 0.35 * line(c + 0.85, 0.35)
shape = pseudo_modulation(B, shape / np.ptp(shape), 0.1)

def measure(dose, name):
    amp = 100 * (1 - np.exp(-(dose + 40) / 1200)) * rng.normal(1, 0.02)
    scans = amp * shape + 3.0 * rng.standard_normal((4, B.size))
    return Spectrum(B, scans, name=name, freq_GHz=FREQ, power_mW=19.0)

doses = np.array([0, 25, 50, 100, 200, 400, 800, 1600])
spectra = [measure(d, f"{d} Gy") for d in doses]
natural_again = measure(0, "0 Gy, second measurement")

plot.plot_spectra(spectra, [f"{d} Gy" for d in doses], window=(331, 342), title="Simulated dose series");

## 2. Intensities and dose response

Each intensity is measured inside an **intensity window**, by default 100 G
around g = 2.0023 (change it with `IntensityWindow(width, unit, center_g=...)`).
A template fit gives the amplitude with its error, from noise taken outside
the window, and `p_noise`, the probability that noise alone gives that
amplitude. The two measurements of the natural aliquot are averaged
(`combined_intensity`).

In [ ]:
window = IntensityWindow()                            # 100 G around g = 2.0023
print("window:", window.describe(FREQ))
template = (B, shape)

res = [combined_intensity([spectra[0], natural_again], template=template, window=window)]
res += [intensity(s, template=template, window=window) for s in spectra[1:]]
for d, r in zip(doses, res):
    print(f"{d:5d} Gy  I = {r.value:6.2f} ± {r.sigma:4.2f}   p_noise = {r.p_noise:.3f}")

amps = np.array([r.value for r in res])
errs = np.array([r.sigma for r in res])
drc = fit_dose_response(doses, amps, "SSE", sigma=errs)
print(drc.summary())
plot.plot_dose_response(drc);

## 3. Age

With measured sediment: `sediment = analyse_files(...).sediment(water=...)`
(see the user guide).

In [ ]:
tooth = ToothSample(
    De=(drc.De, drc.De_sigma),
    enamel_U=(0.6, 0.06), dentine_U=(12.0, 1.2),
    sediment=Sediment(U=(2.0, 0.1), Th=(7.0, 0.3), K=(1.2, 0.05), water=(0.15, 0.05)),
    beta=ToothLayers(enamel_um=(1100, 100), strip_outer_um=(50, 20), strip_inner_um=(50, 20)),
    cosmic=cosmic_dose_rate(depth_m=1.5, density=1.9, lat_deg=4.6, lon_deg=-74.1, altitude_m=2600),
    uptake_enamel=LinearUptake(), uptake_dentine=LinearUptake(),
)
result = tooth.age()
print(result.summary())
mc = tooth.age_mc(n=1000, seed=2)
print(mc.summary())

fig, (a, b) = plt.subplots(1, 2, figsize=(12, 3.8))
plot.plot_dose_rate(result, ax=a)
plot.plot_age_distribution(mc, ax=b)
fig.tight_layout()

## 4. A wetter past

The sediment water (and the burial depth, through `cosmic_history`) can change
with time. Here the site had 30 % water before 20 ka instead of today's 15 %.

In [ ]:
wet_past = Sediment(U=(2.0, 0.1), Th=(7.0, 0.3), K=(1.2, 0.05),
                    water=History([(0.15, 0.05), (0.30, 0.08)], breaks=[20]))
tooth_wet = dataclasses.replace(tooth, sediment=wet_past)
print(f"constant water: {tooth.age().age:6.1f} ka")
print(f"wetter past:    {tooth_wet.age().age:6.1f} ka")

More: [user guide](https://github.com/MiguelGamezL/EPRdating/blob/main/docs/guide.md) ·
[figures](https://github.com/MiguelGamezL/EPRdating/blob/main/docs/plotting.md) ·
[API reference](https://github.com/MiguelGamezL/EPRdating/blob/main/docs/api/index.md)